# VL12 - LLMs & Prompting
In this lecture we will explore some basics about prompting LLMs, reflecting on the impact of some main prompting techniques: zero-shot, few-shot, Chain-of-Thought (CoT). We will also reflect on the areas where prompting is effective.

## 1. Setting up our environment
In this lab, we will interact with hosted large language models (LLMs), in particular with OpenAI models.

However, due to restrictions in the YourAI Jupyter cluster, direct API calls (HTTP or sockets) from notebooks are not allowed.

To work around this limitation, we use a file-based client–server architecture:
- A server runs locally in the terminal and communicates with the hosted LLM API.
- A client runs inside the Jupyter notebook.

The client and server exchange requests and responses via files (read/write), instead of network calls. \

#### Installing the dependencies
First, install the dependencies that our LLMServer and LLMClient will need.

````sh
$ pip install flask requests
```` 
#### Starting the LLM server

Next, navigate to the `Lab11` folder and start the server:

````sh
$ python llm_server_file.py
```` 
This will create a folder within `Lab11` called `llm_bridge` where the requests and responses between server and client will be read/written.

#### Initializing and testing the client

In [ ]:
from llm_client_file import LLMClient

client = LLMClient(base_dir="./llm_bridge", timeout_s=30)

In [ ]:
print(client.prompt(
    "Give me one sentence that explains what prompting is.",
    model="gpt-4o-mini", # by default
    temperature=0.2,
    max_output_tokens=80,
    instructions="You are a concise teaching assistant."
))

## 2. Toy Dataset
One of the prominent tasks we will explore is sentiment analysis. Below is a toy dataset we will use in our prompting tests.

In [ ]:
movie_reviews = [
    {"text": "I loved this movie. The story was engaging and the acting was great.", "sentiment": "positive"},
    {"text": "A wonderful film with beautiful visuals and a strong emotional impact.", "sentiment": "positive"},
    {"text": "The movie was entertaining and kept me interested until the end.", "sentiment": "positive"},
    {"text": "Great performances and a solid plot. I would definitely watch it again.", "sentiment": "positive"},
    {"text": "An excellent movie with memorable characters and a satisfying ending.", "sentiment": "positive"},

    {"text": "The movie was okay, but nothing special really happened.", "sentiment": "neutral"},
    {"text": "It was an average film. Some parts were good, others were boring.", "sentiment": "neutral"},
    {"text": "The story was predictable, but the acting was fine.", "sentiment": "neutral"},
    {"text": "Not bad, not great. It was just a normal movie night.", "sentiment": "neutral"},
    {"text": "The movie had a few good moments, but overall it was forgettable.", "sentiment": "neutral"},

    {"text": "I did not like this movie at all. It was boring and too long.", "sentiment": "negative"},
    {"text": "The plot made no sense and the acting was terrible.", "sentiment": "negative"},
    {"text": "A disappointing film with weak characters and poor dialogue.", "sentiment": "negative"},
    {"text": "The movie was confusing and not enjoyable.", "sentiment": "negative"},
    {"text": "I regret watching this movie. It was a waste of time.", "sentiment": "negative"}
]

## 3. Prompting

### 3.1 Zero-shot sentiment classification

**Goal:** Classify the sentiment of each review as `positive`, `neutral`, or `negative` with **no examples**.

In [ ]:
ZERO_SHOT_SENTIMENT_TMPL = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Return ONLY a JSON object with:
- sentiment
- confidence (0 to 1)

Review:
"{review_text}"
"""

In [ ]:
review = movie_reviews[0]
prompt = ZERO_SHOT_SENTIMENT_TMPL.format(review_text=review["text"])

response = client.prompt(
    prompt,
    #model="gpt-3.5-turbo",
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are a precise and concise sentiment classifier."
)

print (review["text"], "\n")
print (response)

### 3.2 Few-shot sentiment classification

**Goal:** Improve consistency by giving the model a few labeled examples before the real reviews.


In [ ]:
FEW_SHOT_SENTIMENT_TEMPLATE = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Return ONLY a JSON object with:
- sentiment
- confidence (0 to 1)

Examples:
Review: "I absolutely loved it. Great acting and a moving story."
Answer: {{"sentiment":"positive","confidence":0.92}}

Review: "It was fine, but I probably won’t remember it tomorrow."
Answer: {{"sentiment":"neutral","confidence":0.71}}

Review: "The plot was messy and the movie was painfully boring."
Answer: {{"sentiment":"negative","confidence":0.90}}

Now classify this review:
"{review_text}"
"""

In [ ]:
# Few-shot
review = movie_reviews[0]

few_shot_prompt = FEW_SHOT_SENTIMENT_TEMPLATE.format(review_text=review["text"])
few_shot_response = client.prompt(
    few_shot_prompt,
    #model="gpt-3.5-turbo",
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are a precise and consistent sentiment classifier."
)

print (review["text"], "\n")
print(few_shot_response)

### 3.3 Chain-of-Thought (CoT) sentiment classification

**Goal:** Encourage more careful reasoning by asking for step-by-step thinking, but still keep the output structured.

In [ ]:
COT_SENTIMENT_TEMPLATE = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Think step by step about:
1) Overall tone
2) Key words or phrases that signal sentiment
3) Any mixed or neutral signals

After reasoning, return ONLY a JSON object with:
- sentiment
- confidence (0 to 1)
- explanation

Review:
"{review_text}"
"""

In [ ]:
review = movie_reviews[0]

cot_prompt = COT_SENTIMENT_TEMPLATE.format(review_text=review["text"])
cot_response = client.prompt(
    cot_prompt,
    #model="gpt-3.5-turbo",
    temperature=0.2,
    max_output_tokens=200,
    instructions="You are a careful analyst who reasons step by step before answering."
)

print (review["text"], "\n")
print(cot_response)

### 3.4 Pushing LLMs in some "reasoning" tasks
Sentiment analysis is quite a standard tasks, so we expect LLMs be quite good at it. Let's push the models a bit further and see if prompting for CoT makes a difference. 

In [ ]:
reasoning_tasks = [
    {
        "question": (
            "Today, Hannah went to the soccer field. Between what times could she have gone?\n\n"
            "We know that:\n"
            "- Hannah woke up at 6:30am.\n"
            "- Hannah ate breakfast for 1 hour after waking up.\n"
            "- Hannah worked from 8:00am to 12:00pm.\n"
            "- Hannah met a friend for 1 hour sometime after work.\n"
            "- The soccer field opened at 6:00am.\n"
            "- The soccer field closed at 6:00pm.\n"
            "- Hannah stayed at the soccer field for at least 1 hour.\n\n"
            "Options:\n"
            "(A) 6:00am – 7:00am\n"
            "(B) 7:00am – 8:00am\n"
            "(C) 12:00pm – 1:00pm\n"
            "(D) 1:30pm – 2:30pm\n"
            "(E) 5:00pm – 5:30pm\n\n"
            "Output only the letter of the correct option."
        ),
        "gold": "E"
    },
    {
        "question" : (
            "I went to the market and bought 10 apples.\n"
            "I gave 2 apples to the neighbor and 2 to the repairman. \n"
            "I then went and bought 5 more apples and ate the same number of apples I gave to the neighbor.\n" 
            "How many apples did I remain with?"
        ),
        "gold" : "9"
    }
]

In [ ]:
ZERO_SHOT_REASONING_TEMPLATE = """
Solve the task below.

{task_text}
"""

COT_REASONING_TEMPLATE = """
You are a careful reasoning assistant.

Solve the task below.

Think step by step. Double-check your reasoning.
Then output ONLY the final answer in the requested format.

{task_text}
"""

#### Zero-shot in a reasoning task

In [ ]:
task = reasoning_tasks[1]
prompt = ZERO_SHOT_REASONING_TEMPLATE.format(task_text=task["question"])

response = client.prompt(
    prompt,
    #model="gpt-3.5-turbo",
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are concise. Follow the task instructions exactly.."
)

print (response)

#### CoT in a reasoning task

In [ ]:
prompt = COT_REASONING_TEMPLATE.format(task_text=task["question"])

response = client.prompt(
    prompt,
    model="gpt-3.5-turbo",
    temperature=0.2,
    max_output_tokens=250,
    instructions="You are concise. Follow the task instructions exactly.."
)

print (response)

### 3.5 Reflection
Do modern LLMs benefit from prompting?

**In particular, where does few-shot matter?**
- We have seen that in modern models (such as GPT-4o-mini) having few shots examples can be useful to format output
- Can be useful also to disambiguate the natural langauge instructions (examples are sometimes easier to specify, and be precise about)
- Following specific domain-specific or project-specific conventions or labels (here some examples or demonstrations are useful) 
- To handle edge cases, especially when we identify examples or scenarios where the model is failing - we can add those examples to reinforce correct behavior

**What about CoT?**
- not universally necessary, especially in new "reasoning" models that already have internal multi-step reasoning without being ask.
- still very relevant in non-reasoning models and lightweight models
- Also relevant for "transparency" or cases where explainability is important. Explainability and inspectable decisions are a big thing now!

## 4. Using an LLM to solve NLP tasks
We often interact with LLMs through chat interfaces, but an important use case is integrating LLMs programmatically to solve classic NLP tasks.

Using our `LLMClient`, we can evaluate an LLM on a sentiment classification task, in the same way we would evaluate traditional NLP models. We apply a prompt to each example in a small sentiment dataset and compare the model’s predictions to the gold labels.

This allows us to treat the LLM as a task-solving component, rather than a conversational agent, and to analyze its performance using standard evaluation metrics.

Let’s see how it performs on our toy sentiment dataset.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def extract_label(text: str):
    """
    Very simple parser:
    - expects the model to output 'positive', 'neutral', or 'negative'
    - falls back to searching for these words in the output
    """
    t = (text or "").strip().lower()
    for lbl in ["positive", "neutral", "negative"]:
        if t == lbl:
            return lbl
    for lbl in ["positive", "neutral", "negative"]:
        if lbl in t:
            return lbl
    return "invalid"

def evaluate_sentiment(dataset, prompt_template, instructions):
    y_true = []
    y_pred = []

    for ex in dataset:
        prompt = prompt_template.format(review_text=ex["text"])

        response = client.prompt(
            prompt,
            temperature=0.2,
            max_output_tokens=30,
            instructions=instructions
        )

        pred = extract_label(response)
        y_true.append(ex["sentiment"])
        y_pred.append(pred)

    return y_true, y_pred


In [ ]:
# Cell 3 — Run + print metrics (pick one template)

y_true, y_pred = evaluate_sentiment(
    movie_reviews,
    ZERO_SHOT_SENTIMENT_TMPL,
    instructions="You are a precise sentiment classifier."
)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred, labels=["positive","neutral","negative","invalid"]))
print("\nReport:\n", classification_report(y_true, y_pred, labels=["positive","neutral","negative","invalid"]))


### Reflection
LLMs perform extremely well on classic NLP tasks like sentiment analysis, often matching or exceeding traditional models in accuracy.
However, this comes at the cost of speed, cost, and control. Dedicated models are still preferable when tasks are stable, high-volume, or latency-sensitive.